# Feature Engineering

### 
* Scaling,
* Normalization,
* Creation of new columns/attributes

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import categorize_title, map_to_sub_zone, clean_title, cat1


#import inflect
#import re

from warnings import filterwarnings
filterwarnings('ignore')

#style.set_properties(**{'max': {'width': 100}})
sns.set_style('whitegrid')

pd.set_option('display.expand_frame_repr',False)
pd.set_option('display.max_colwidth',None)



%matplotlib inline

In [19]:
df1 = pd.read_csv("Data_for_EDA.csv",index_col = 0)
df1.columns = [ col.lower().strip() for col in df1.columns ] 
for col in ['issue_d', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']:
    df1[col] = pd.to_datetime(df1[col], format='%b-%y', errors='coerce')

In [20]:
numeric = df1.select_dtypes(include = ['float64','int64'])
cat = df1.select_dtypes(include = ['object'])

In [21]:
new = df1.copy()

In [22]:
cat.columns

Index(['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'loan_status', 'pymnt_plan',
       'url', 'purpose', 'title', 'zip_code', 'addr_state',
       'initial_list_status', 'application_type'],
      dtype='object')

In [23]:
# Feature Engineering
#df
new = df1.copy()

## term
df1['term'] = df1['term'].apply(lambda x: x.strip().replace('months',''))
df1['term'] = df1['term'].apply(lambda x: x.strip())


## emp_length


df1['emp_length'].unique()
df1['emp_length'] = df1['emp_length'].str.extract(r'(\d+)')


##  addr_state

### Importing files containg zone and the states in abbreviation from

zo = pd.read_csv('zones.csv',)
zo.columns = [x.lower().strip() for x in zo.columns]
zo.head()

# creating a dictionary with zone as key, and the states falling into that zone as their values,
#since a zone can have multiple states in them, values are getting stored as a list.
zo_states_dict = {}

for index, row in zo.iterrows():
    zo_name = row['region']
    states = [row['postal abbreviation']]  # list with the state name
    
    if zo_name in zo_states_dict:
        zo_states_dict[zo_name].extend(states)
    else:
        zo_states_dict[zo_name] = states


#creating a new column ,'region' using reverse dictionary and mapping it to the addr_state column present in our dataset

state_to_region = {state: region for region, states in zo_states_dict.items() for state in states}

df1['region'] = df1['addr_state'].map(state_to_region)


df1['region'].value_counts()

df1.sample(2)



##  zip_code


,unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,target_loan_status,region
170352,287878.0,29103714,31636890,20000,20000,20000.0,60,13.98,465.16,C,...,NaT,0,1,INDIVIDUAL,0,0.0,79861.0,46300.0,True,Northeast
237372,182735.5,49036576,52325297,9375,9375,9375.0,36,15.61,327.80,D,...,NaT,0,1,INDIVIDUAL,0,0.0,5355.0,29000.0,True,South


In [25]:
##  zip_code

df1['zip_code'].value_counts().head(3)
df1['zip_code'] = df1['zip_code'].str.replace('x','')
df1['zip_code'].unique()[:10]


nd = new_df.copy()


#defining the code to categorise zip_codes into subzones,

def map_to_sub_zone(row):
    region_name = None
    for region, states in zo_states_dict.items():
        if row['addr_state'] in states:
            region_name = region
            break
    
    if region_name:
        zone_range = 250  
        sub_zone = (int(row['zip_code']) // zone_range) + 1  #creating 4 sub zones for each value predsnt in region.
        return f"{region_name} - Sub-Zone {sub_zone}"
    
    return None

df1['sub-regions'] = df1.apply(map_to_sub_zone, axis=1)

df1['sub-regions'].value_counts()

df1.sample(2)


df1.loc[df1[df1['sub-regions'].eq('West - Sub-Zone 4')].index,
           ['zip_code','addr_state','region','sub-regions']].sample(5)



,zip_code,addr_state,region,sub-regions
205124,945,CA,West,West - Sub-Zone 4
155286,945,CA,West,West - Sub-Zone 4
208231,941,CA,West,West - Sub-Zone 4
134276,967,HI,West,West - Sub-Zone 4
197482,945,CA,West,West - Sub-Zone 4


In [26]:
##  emp_title and addr_state 


df1['emp_title'].nunique()
df1['emp_title'].value_counts().sort_values(ascending = False).head(20)
df1['emp_title'].replace(np.nan, 'null', inplace = True)

df1[df1['emp_title'].str.contains(r'[^\w\s]', regex=True)]['emp_title'].sample(5)


# checking for special characters
for i in df1['emp_title']:
    for j in i:
        if j in ['#','*','?']:
            print(i)


Wrenshall Public School District #100
Laborers Union Local #121
Allied Waste Services #922
E*Trade
Johnson County Fire District # 2
Sweetwater School District #1
North Greene Unit District #3
McCord Rural Water District #3
The Catfish Hole #3
laborers local  #91
Greenburgh Central School District #7
North Branch Fire District #1
local union #3 IBEW
Johnson County School District #1
1800-got-junk?
E*Trade Financial
Woodridge School District #68
E*TRADE Financial
Laramie County Schools #1
Adams County School District #50
IBEW Local #3
Nogales Unified School District #1
Lockwood School District #26
Safeway Store #1160
Snohomish County Fire District #5
Local Union #3  Welsbac Electric
nhaj l.c. dba bestop # 4
ibew local union #3
Page Unified School Dist. #8
ISD #11
Laramie County School District #1
E*Trade Financial
The Cirignano Limited Partnership #2
IBEW #6 EISB
SIGN*A*RAMA
SIGN*A*RAMA
Walmart Supercenter #3391
Elmhurst Community School Dist. #205
El Paso School District #11
Bay Area A?

In [16]:
import inflect

In [27]:
df1['emp_clean_title'] = df1['emp_title'].apply(clean_title)

df1['emp_clean_title'] = df1['emp_clean_title'].apply(lambda x:x.lower())

df1['emp_clean_title'].nunique()


# we can see clearly that number of unique values in emp_title column gets down drastically upon treating the values, let's explore more

df1['emp_clean_title'].replace('registerd nurse','registered nurse',inplace = True)

df1['emp_clean_title'].value_counts().head(10)


# Function to categorize the titles
#def categorize_title(title):

emp_clean_title
null                14010
manager              2720
teacher              2421
supervisor           1304
registered nurse     1236
sales                1056
driver                988
rn                    971
us army               835
project manager       819
Name: count, dtype: int64

In [18]:
!pip install inflect

In [28]:
# Create a df1 column 'category' based on the words present in the title
df1['category'] = df1['emp_clean_title'].apply(lambda x: categorize_title(x.lower()))


df1['category'].value_counts().lt(5).sum()
df1['category'].nunique()

### checkpoint
az = new.copy()


threshold = 100

# Count the occurrences of each unique value in the column
value_counts = df1['category'].value_counts()
less_than_threshold_values = value_counts[value_counts < threshold].index
df1['category1'] = df1['category'].where(~df1['category'].isin(less_than_threshold_values), 'Others')


df1['category1'].replace('null',np.nan,inplace=True)

df1.fillna(df1['category1'].mode()[0],inplace = True)


df1['category1'].replace(['united states air force','usaf'],'us air force',inplace = True)
df1['category1'].replace('jpmorgan chase','jp morgan chase',inplace = True)



df1.fillna(df1['category1'].mode()[0],inplace = True)

df1['category1'].replace('vp','vice president',inplace = True)


def cat1(title):
    title = title.lower()


df1['cat'] = df1['category1'].apply(lambda x: cat1(x.lower()))

#df1.to_csv('lending_clubdata_Imbalanced.csv')

In [30]:
df1.select_dtypes('object')

,term,grade,sub_grade,emp_title,emp_length,home_ownership,verification_status,issue_d,loan_status,pymnt_plan,...,initial_list_status,last_pymnt_d,last_credit_pull_d,application_type,region,sub-regions,emp_clean_title,category,category1,cat
0,36,B,B2,null,10,RENT,Verified,Others,Fully Paid,n,...,f,Others,Others,INDIVIDUAL,West,West - Sub-Zone 4,null,null,Others,None
1,60,C,C4,Ryder,1,RENT,Source Verified,Others,Default,n,...,f,Others,Others,INDIVIDUAL,South,South - Sub-Zone 2,ryder,ryder,Others,None
2,36,C,C5,null,10,RENT,Not Verified,Others,Fully Paid,n,...,f,Others,Others,INDIVIDUAL,Midwest,Midwest - Sub-Zone 3,null,null,Others,None
3,36,C,C1,AIR RESOURCES BOARD,10,RENT,Source Verified,Others,Fully Paid,n,...,f,Others,Others,INDIVIDUAL,West,West - Sub-Zone 4,air resources board,air resources board,Others,None
4,36,A,A4,Veolia Transportaton,3,RENT,Source Verified,Others,Fully Paid,n,...,f,Others,Others,INDIVIDUAL,West,West - Sub-Zone 4,veolia transportaton,veolia transportaton,Others,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254185,36,D,D2,supervisor,10,MORTGAGE,Verified,Others,Default,n,...,f,Others,Others,INDIVIDUAL,West,West - Sub-Zone 4,supervisor,supervisors,supervisors,None
254186,36,A,A1,Coordinator of RSVP,1,RENT,Not Verified,Others,Fully Paid,n,...,w,Others,Others,INDIVIDUAL,South,South - Sub-Zone 2,coordinator of rsvp,coordinator,coordinator,None
254187,36,D,D3,Painter,2,RENT,Source Verified,Others,Fully Paid,n,...,f,Others,Others,INDIVIDUAL,South,South - Sub-Zone 2,painter,painter,Others,None
254188,36,B,B1,Lead Custodian,10,MORTGAGE,Verified,Others,Fully Paid,n,...,f,Others,Others,INDIVIDUAL,West,West - Sub-Zone 4,lead custodian,lead custodian,Others,None


In [31]:
df1.rename(columns = {'cat':'emp_title_fe'},inplace = True)

df1.drop(['emp_title','emp_clean_title','category','category1'],
        axis = 1, inplace = True)

df1.shape

df1.head(2)


#df1.to_csv('lending_club_Imbalance.csv')

,unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,policy_code,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,target_loan_status,region,sub-regions,emp_title_fe
0,0.0,1077501,1296599,5000,5000,4975.0,36,10.65,162.87,B,...,1,INDIVIDUAL,0,0.0,80760.5,22300.0,True,West,West - Sub-Zone 4,None
1,1.0,1077430,1314167,2500,2500,2500.0,60,15.27,59.83,C,...,1,INDIVIDUAL,0,0.0,80760.5,22300.0,False,South,South - Sub-Zone 2,None


In [32]:
new_df = df1.copy()
new_df['title'] = new_df['title'].apply(lambda x: x.lower().strip())


new_df['title'].nunique()

new_df['title'].value_counts().head(30).values.sum()

df1['purpose'].value_counts()


data1 = df1.copy()




In [33]:


df1.drop(['zip_code','addr_state','loan_status'],axis = 1,inplace = True)

cat.drop(['zip_code','addr_state','loan_status'],axis = 1,inplace = True)

cat.shape

df1.shape



(254190, 54)

In [34]:
cat

,term,grade,sub_grade,emp_title,emp_length,home_ownership,verification_status,pymnt_plan,url,purpose,title,initial_list_status,application_type
0,36 months,B,B2,NaN,10+ years,RENT,Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=1077501,credit_card,Computer,f,INDIVIDUAL
1,60 months,C,C4,Ryder,< 1 year,RENT,Source Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=1077430,car,bike,f,INDIVIDUAL
2,36 months,C,C5,NaN,10+ years,RENT,Not Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=1077175,small_business,real estate business,f,INDIVIDUAL
3,36 months,C,C1,AIR RESOURCES BOARD,10+ years,RENT,Source Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=1076863,other,personel,f,INDIVIDUAL
4,36 months,A,A4,Veolia Transportaton,3 years,RENT,Source Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=1075269,wedding,My wedding loan I promise to pay back,f,INDIVIDUAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...
254185,36 months,D,D2,supervisor,10+ years,MORTGAGE,Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=36743377,medical,Medical expenses,f,INDIVIDUAL
254186,36 months,A,A1,Coordinator of RSVP,< 1 year,RENT,Not Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=36231718,debt_consolidation,Debt consolidation,w,INDIVIDUAL
254187,36 months,D,D3,Painter,2 years,RENT,Source Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=36241316,debt_consolidation,Debt consolidation,f,INDIVIDUAL
254188,36 months,B,B1,Lead Custodian,10+ years,MORTGAGE,Verified,n,https://www.lendingclub.com/browse/loanDetail.action?loan_id=36421485,car,Car financing,f,INDIVIDUAL
